# Stage 1B — Pooled Multi-Subject Class Balance Check

**Goal:** confirm that pooling across subjects/runs actually solves the data-starved
"incorrect + high-confidence" cell we found in subject 01 alone (only 5 trials).

**You only need the `EEG_events_sub-XX_run-YY.mat` files for this** — these are tiny
(a few KB each), so uploading/mounting all of them is not a size problem, unlike the raw EEG.

Two ways to get the files in:
- If they're in Google Drive, mount Drive and point `EVENTS_DIR` at the folder (recommended for many files).
- Or upload a handful directly with `files.upload()`.

In [ ]:
# 1. Get the events files in. Try Drive first; fall back to manual upload.
import os, glob, re

USE_DRIVE = True  # set False if you'd rather upload files manually

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    EVENTS_DIR = "/content/drive/MyDrive/PATH_TO_YOUR_EVENTS_FILES"  # <-- EDIT THIS to your actual folder
else:
    from google.colab import files
    print("Upload all EEG_events_sub-*.mat files (multi-select allowed)")
    uploaded = files.upload()
    EVENTS_DIR = "."

event_files = sorted(glob.glob(os.path.join(EVENTS_DIR, "**", "EEG_events_sub-*.mat"), recursive=True))
print(f"Found {len(event_files)} events files")
for f in event_files[:10]:
    print(" ", f)
if len(event_files) > 10:
    print(f"  ... and {len(event_files)-10} more")

In [ ]:
# 2. Load every events file, tag with subject/run parsed from filename, collect into one table
from scipy.io import loadmat
import numpy as np

pattern = re.compile(r"sub-(\d+)_run-(\d+)")

rows = []  # each row: subject, run, confidence, accuracy, choice, dotdirection, tstim, tresp, RT

for fpath in event_files:
    m = pattern.search(os.path.basename(fpath))
    subj = m.group(1) if m else "unknown"
    run  = m.group(2) if m else "unknown"

    ev = loadmat(fpath)
    confidence = ev["confidence"].squeeze()
    accuracy   = ev["accuracy"].squeeze()
    choice     = ev["choice"].squeeze()
    dotdirection = ev["dotdirection"].squeeze()
    tstim = ev["tstim"].squeeze()
    tresp = ev["tresp"].squeeze()
    RT    = ev["RT"].squeeze()

    n = len(confidence)
    for i in range(n):
        rows.append({
            "subject": subj, "run": run,
            "confidence": int(confidence[i]), "accuracy": int(accuracy[i]),
            "choice": int(choice[i]), "dotdirection": int(dotdirection[i]),
            "tstim": float(tstim[i]), "tresp": float(tresp[i]), "RT": float(RT[i]),
        })

print(f"Total pooled trials across all loaded files: {len(rows)}")

In [ ]:
# 3. Per-subject trial counts (sanity check nothing loaded oddly / no subject is an outlier)
from collections import defaultdict

per_subject_n = defaultdict(int)
for r in rows:
    per_subject_n[(r["subject"], r["run"])] += 1

print(f"{'subject':>8} {'run':>5} {'n_trials':>9}")
for (s, ru), n in sorted(per_subject_n.items()):
    print(f"{s:>8} {ru:>5} {n:>9}")

n_files_loaded = len(per_subject_n)
print(f"\n{n_files_loaded} subject/run combinations loaded, {len(rows)} trials total")

In [ ]:
# 4. Pooled class balance: choice, accuracy
confidence_all = np.array([r["confidence"] for r in rows])
accuracy_all   = np.array([r["accuracy"] for r in rows])
choice_all     = np.array([r["choice"] for r in rows])

n_total = len(rows)

print("=== Pooled choice balance ===")
vals, counts = np.unique(choice_all, return_counts=True)
for v, c in zip(vals, counts):
    print(f"  choice={v}: {c} trials ({100*c/n_total:.1f}%)")

print("\n=== Pooled accuracy balance ===")
vals, counts = np.unique(accuracy_all, return_counts=True)
for v, c in zip(vals, counts):
    print(f"  accuracy={v}: {c} trials ({100*c/n_total:.1f}%)")

In [ ]:
# 5. THE key check: pooled 4-way accuracy x confidence table (same Low/High binning as before)
sorted_conf = np.sort(np.unique(confidence_all))
low_labels  = sorted_conf[:3]
high_labels = sorted_conf[-3:]
print("Low confidence ratings:", low_labels, " High confidence ratings:", high_labels)

is_low  = np.isin(confidence_all, low_labels)
is_high = np.isin(confidence_all, high_labels)
is_med  = ~is_low & ~is_high

print(f"\nLow: {is_low.sum()}  Medium: {is_med.sum()}  High: {is_high.sum()}  (of {n_total} total)")

print(f"\n{'':>20} {'Low conf':>10} {'High conf':>10}")
for acc_val, acc_label in [(1, 'Correct'), (0, 'Incorrect')]:
    acc_mask = accuracy_all == acc_val
    n_low  = int(np.sum(acc_mask & is_low))
    n_high = int(np.sum(acc_mask & is_high))
    print(f"{acc_label:>20} {n_low:>10} {n_high:>10}")

n_incorrect_high = int(np.sum((accuracy_all == 0) & is_high))
n_incorrect_low  = int(np.sum((accuracy_all == 0) & is_low))
n_correct_high   = int(np.sum((accuracy_all == 1) & is_high))
n_correct_low    = int(np.sum((accuracy_all == 1) & is_low))

print(f"\n>>> Pooled incorrect+high-confidence ('overconfidence') trials: {n_incorrect_high}")
if n_incorrect_high < 40:
    print(">>> Still thin for a full per-trial decoder. Consider grouping by subject ")
    print(">>> (subject-level features / mixed-effects) rather than pooling as if trials were iid,")
    print(">>> since trials from the same subject aren't independent.")
else:
    print(">>> Reasonable trial count to attempt single-trial decoding, with appropriate")
    print(">>> cross-validation that respects subject grouping (e.g. leave-one-subject-out).")

In [ ]:
# 6. How many distinct subjects contribute to the overconfidence cell?
# (Important: if only 2-3 subjects supply almost all incorrect+high trials, pooled
#  decoding could just be learning to tell those subjects apart, not confidence.)
subj_of_row = np.array([r["subject"] for r in rows])
mask_overconf = (accuracy_all == 0) & is_high
subjects_in_cell = np.unique(subj_of_row[mask_overconf])
print(f"Subjects contributing to the incorrect+high-confidence cell: {len(subjects_in_cell)}")
print(subjects_in_cell)

print("\nPer-subject count in this cell:")
vals, counts = np.unique(subj_of_row[mask_overconf], return_counts=True)
for v, c in zip(vals, counts):
    print(f"  subject {v}: {c} trials")

## What to send back

Paste the full printed output of cells 1 (file count), 3 (per-subject trial counts),
4, 5, and 6. Cell 5 is the critical one — it tells us whether pooling actually rescues
the overconfidence analysis, and cell 6 tells us whether that rescue is trustworthy
(spread across many subjects) or fragile (dominated by 1-2 subjects).